In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from fraud_detection.config import load_config
from fraud_detection.data.load import load_interim
from fraud_detection.data.split import assign_split
from fraud_detection.evaluation.metrics import evaluate
from fraud_detection.features.preprocessing import TARGET, Preprocessor
from fraud_detection.models.xgb import train_from_config

In [ ]:
df = load_interim()

In [ ]:
data_cfg = load_config("data")
xgb_cfg = load_config("xgboost")

split = assign_split(df, **data_cfg["split"])
train, val = df[split == "train"], df[split == "val"]

prep = Preprocessor(**data_cfg["preprocessing"]).fit(train)

In [ ]:
X_train, y_train = prep.transform(train), train[TARGET]
X_val, y_val = prep.transform(val), val[TARGET]

In [ ]:
print(X_train.shape, X_val.shape)
print(len(prep.v_kept_), "colonnes V conservées")

In [ ]:
import numpy as np
import xgboost as xgb

xgb.XGBClassifier(device="cuda", n_estimators=10).fit(
    np.random.rand(1000, 5), np.random.randint(0, 2, 1000)
)
print("XGBoost GPU OK")

In [ ]:
cfg = load_config("xgboost")
model, params, study = train_from_config(X_train, y_train, X_val, y_val, cfg)

val_scores = model.predict_proba(X_val)[:, 1]
pd.Series(evaluate(y_val, val_scores)).round(4)

In [ ]:
import yaml
from fraud_detection.data.load import PROJECT_ROOT
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
)

if study is not None:
    plot_optimization_history(study)
    plot_param_importances(study)
    with open(
        PROJECT_ROOT / "configs" / "xgboost_tuned.yaml", "w", encoding="utf-8"
    ) as f:
        yaml.safe_dump({"params": params, "tuning": {"enabled": False}}, f)

In [ ]:
import matplotlib.pyplot as plt

importance = pd.Series(
    model.get_booster().get_score(importance_type="gain")
).sort_values()
importance.tail(20).plot(
    kind="barh", figsize=(8, 7), title="Top 20 des variables (gain)"
)
plt.tight_layout()
plt.savefig(
    PROJECT_ROOT / "reports" / "figures" / "importance_xgb_tuned.png",
    dpi=120,
    bbox_inches="tight",
)

In [ ]:
name = "xgb_tuned" if study is not None else "xgb_baseline"
model.save_model(PROJECT_ROOT / "models" / f"{name}.json")
print(f"Modèle sauvegardé : {name}.json")

In [ ]:
print(df["V257"].isna().mean())
pd.crosstab(df["ProductCD"], df["V257"].notna())

In [ ]:
sub = df[df["V257"].notna()]
sub.groupby(pd.qcut(sub["V257"], 10, duplicates="drop"), observed=True)["isFraud"].agg(
    ["size", "mean"]
)

**Modèle not-tuned**

| Modèle | PR-AUC | ROC-AUC | Précision@2% | Rappel@2% |
|---|---|---|---|---|
| XGBoost baseline | 0,606 | 0,922 | 0,768 | 0,453 |

**Modèle tuned**

| Modèle | PR-AUC | ROC-AUC | Précision@2% | Rappel@2% |
|---|---|---|---|---|
| XGBoost tuned | 0.607 | 0.924 | 0.765 | 0.451 |